In [ ]:
import os
import sys
import time
import gc
from typing import List, Tuple
import pickle
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
project_path = os.path.abspath(os.path.dirname(os.getcwd()))
sys.path.append(project_path)
from src import *

In [ ]:
experiment =  'stella_maris_pretrain_56channel_final'
experiment_path = os.path.join(project_path, 'results', experiment)
spike_path = os.path.join(experiment_path, 'spikes')
spike_list = os.listdir(spike_path)

In [ ]:
all_windows = []
subject_labels = {}

for spike_file in spike_list:
    if not spike_file.endswith(".npz"):
        continue

    subj = spike_file.replace("_spikes.npz","")
    spikes = np.load(os.path.join(spike_path, spike_file))

    all_spikes = spikes["concat_raw"]
    labels_raw = spikes["labels_raw"]
    window_lengths = spikes["window_lengths"]
    if 'SC' in spike_file and int(labels_raw[0]) == 1:
        print(spike_file)
        continue
    subject_labels[subj] = int(labels_raw[0])
    ptr = 0
    for L in window_lengths:
        w = all_spikes[ptr:ptr+L,:]
        y = int(labels_raw[ptr])
        all_windows.append((w,y,subj))
        ptr += L


In [ ]:
subject_list = list(subject_labels.keys())
subject_y = np.array([subject_labels[s] for s in subject_list])

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
blocks = [test_idx for _,test_idx in skf.split(subject_list, subject_y)]


In [ ]:
fold = 0
test_block = blocks[fold]
val_block  = blocks[(fold+1)%5]
train_blocks = [blocks[(fold+i)%5] for i in range(2,5)]

In [ ]:
train_idx = np.concatenate(train_blocks)

train_subjects = [subject_list[i] for i in train_idx]
val_subjects   = [subject_list[i] for i in val_block]
test_subjects  = [subject_list[i] for i in test_block]

train_set = [x for x in all_windows if x[2] in train_subjects]
val_set   = [x for x in all_windows if x[2] in val_subjects]
test_set  = [x for x in all_windows if x[2] in test_subjects]

In [ ]:
def extract_subject_features_from_spikes(subject_spikes_paths):
    """
    Ricostruisce subject_features esattamente come in pretrain.py
    usando SOLO i file .npz (filter_raw, source_raw, window_lengths).

    Output:
        subject_features_new : dict[subj] = feature_vector (2N,)
    """

    subject_features_new = {}

    for subj, path in subject_spikes_paths.items():

        data = np.load(path)
        filter_raw = data["filter_raw"]          # (T_tot, N)
        source_raw = data["source_raw"]          # (T_tot, N)
        window_lengths = data["window_lengths"]  # (W,)
        labels_raw = data["labels_raw"]          # (T_tot,)  # non lo usiamo

        # ricostruzione identica al pretrain
        ptr = 0
        window_feats = []

        for L in window_lengths:
            f_mean = filter_raw[ptr:ptr+L].mean(axis=0)    # (N,)
            s_mean = source_raw[ptr:ptr+L].mean(axis=0)    # (N,)
            feat_window = np.concatenate([f_mean, s_mean]) # (2N,)
            window_feats.append(feat_window)
            ptr += L

        # media su tutte le finestre del soggetto
        subject_features_new[subj] = np.mean(window_feats, axis=0)

    return subject_features_new

def compute_global_report(confusion_matrices):
    cm = np.sum(confusion_matrices, axis=0)
    tn, fp, fn, tp = cm.ravel()

    precision_pos = tp / (tp + fp) if tp + fp > 0 else 0
    recall_pos    = tp / (tp + fn) if tp + fn > 0 else 0
    f1_pos = 2 * precision_pos * recall_pos / (precision_pos + recall_pos) if precision_pos + recall_pos > 0 else 0

    precision_neg = tn / (tn + fn) if tn + fn > 0 else 0
    recall_neg    = tn / (tn + fp) if tn + fp > 0 else 0
    f1_neg = 2 * precision_neg * recall_neg / (precision_neg + recall_neg) if precision_neg + recall_neg > 0 else 0

    accuracy = (tp + tn) / (tp + tn + fp + fn)

    report = {
        "accuracy": accuracy,
        "class_0": {
            "precision": precision_neg,
            "recall": recall_neg,
            "f1-score": f1_neg
        },
        "class_1": {
            "precision": precision_pos,
            "recall": recall_pos,
            "f1-score": f1_pos
        }
    }

    return cm, report


In [ ]:
import os
import pickle

load_path = os.path.join(experiment_path, "subject_features.pkl")
if not os.path.exists(load_path):
    load_path = os.path.join(project_path, 'results', experiment, "subject_features.pkl")

with open(load_path, "rb") as f:
    data_loaded = pickle.load(f)

subject_features = data_loaded["features"]
subject_labels = data_loaded["labels"]
subjects_list  = data_loaded["subjects"]

spike_dir = spike_path   # la tua directory, già definita
subject_spikes_paths = {}
subject_features_dict = {}
for idx, fname in enumerate(spike_list):
    if fname.endswith("_spikes.npz"):
        subj = fname.replace("_spikes.npz", "")
        if subj in ['SC_037', 'SC_068', 'SC_054', 'SC_72', 'SC_86']:
            continue
        subject_spikes_paths[subj] = os.path.join(spike_dir, fname)
     #   subject_features_dict[subj] = subject_features[idx]

# controllo
print("Num subjects:", len(subject_spikes_paths))
print("Esempio:", list(subject_spikes_paths.items())[:5])

subject_features_from_spikes = extract_subject_features_from_spikes(subject_spikes_paths)


In [ ]:
# ===== REBUILD subject_to_spike_windows FROM NPZ =====

subject_to_spike_windows = {}
subject_labels_npz = {}

for subj, path in subject_spikes_paths.items():

    data = np.load(path)
    concat_raw = data["concat_raw"]            # (T_tot, 1008)
    window_lengths = data["window_lengths"]    # array (num_windows,)
    labels = data["labels_raw"]                # (T_tot,)

    subject_labels_npz[subj] = int(labels[0])  # unica label del soggetto

    ptr = 0
    windows = []

    for L in window_lengths:
        w = concat_raw[ptr:ptr+L, :]           # estrai finestra (L × 1008)
        windows.append(w)
        ptr += L

    subject_to_spike_windows[subj] = windows

print("OK ✓ Ricostruite tutte le finestre dai .npz")
print("Numero soggetti:", len(subject_to_spike_windows))
print("Esempio finestre soggetto:", list(subject_to_spike_windows.keys())[0],
      "num_finestra =", len(subject_to_spike_windows[list(subject_to_spike_windows.keys())[0]]))


### Plot audio-only dal segnale filtrato post-unsupervised
Questo plot usa una singola finestra di un soggetto e costruisce un segnale 1D "audio-like" a partire da `filter_raw` (media sui neuroni per time-step).

In [ ]:
# Scegli un soggetto e una finestra
subj_example = sorted(subject_spikes_paths.keys())[0]
window_idx = 0

npz_path = subject_spikes_paths[subj_example]
data = np.load(npz_path)

filter_raw = data["filter_raw"]          # (T_tot, N)
source_raw = data["source_raw"]          # (T_tot, N)
window_lengths = data["window_lengths"]  # (W,)

# Ricostruzione puntatore della finestra scelta
ptr = int(np.sum(window_lengths[:window_idx]))
L = int(window_lengths[window_idx])

# Segmenti della finestra
f_win = filter_raw[ptr:ptr+L]   # (L, N)
s_win = source_raw[ptr:ptr+L]   # (L, N)

# Segnali 1D "audio-like": media sui neuroni per ciascun time-step
filter_audio_like = f_win.mean(axis=1)
source_audio_like = s_win.mean(axis=1)

# Normalizzazione robusta

def _norm(x):
    x = np.asarray(x, dtype=np.float64)
    s = np.percentile(np.abs(x), 99.5)
    if s <= 1e-12:
        return x
    return np.clip(x / s, -1.0, 1.0)

filter_audio_like = _norm(filter_audio_like)
source_audio_like = _norm(source_audio_like)

# Stile visuale come notebook precedente (solo dato, no assi)
NAVY  = '#0B1B3B'
TEAL  = '#00C2A8'
AMBER = '#FF8A3C'

fig, ax = plt.subplots(figsize=(12, 3), facecolor=NAVY)
ax.set_facecolor(NAVY)
ax.plot(filter_audio_like, color=TEAL, linewidth=0.7)

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

out_plot = os.path.join(experiment_path, f"audio_only_filter_{subj_example}_w{window_idx}.png")
fig.savefig(out_plot, dpi=300, bbox_inches='tight',
            facecolor=NAVY, edgecolor='none', pad_inches=0.05)
plt.show()

print(f"Soggetto: {subj_example}")
print(f"Finestra: {window_idx}  (L={L})")
print(f"PNG salvato in: {out_plot}")


In [ ]:
# ===== METHOD 0 — FIRING RATE PER SOGGETTO =====

SOGLIA_VAR = 0.85

subjects = sorted(subject_to_spike_windows.keys())

# ⚠️ USA IL DIZIONARIO CORRETTO
y = np.array([subject_labels_npz[s] for s in subjects])

# 1) Calcolo X_raw
X_raw = []

for subj in subjects:
    windows = subject_to_spike_windows[subj]
    fr_list = [w.mean(axis=0) for w in windows]
    feat_subj = np.mean(fr_list, axis=0)   # (1008,)
    X_raw.append(feat_subj)

X_raw = np.array(X_raw)
print("X_raw shape:", X_raw.shape)
e
# 2) PCA COMPLETA
pca = PCA()
#pca.fit(X_raw)

X_pca = pca.fit_transform(X_raw)
var = np.cumsum(pca.explained_variance_ratio_)
k = np.searchsorted(var, SOGLIA_VAR) + 1
print(f"k =", k)
# 3) SVM
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        gamma="scale",
        C=1.0
    ))
])

model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.85, svd_solver="full")),
    ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", gamma="scale", C=1.0))
])

conf_mats=[]
#for tr,ts in skf.split(X_pca,y):
 #   model=SVC(kernel="rbf",probability=True,class_weight="balanced")
 #   model.fit(X_pca[tr],y[tr])
 #   y_pred=model.predict(X_pca[ts])
 #   conf_mats.append(confusion_matrix(y[ts],y_pred))
for tr, ts in skf.split(X_raw, y):
    model.fit(X_raw[tr], y[tr])
    y_pred = model.predict(X_raw[ts])
    conf_mats.append(confusion_matrix(y[ts],y_pred))

cm_tot, rep_tot = compute_global_report(conf_mats)

print(rep_tot)

sns.heatmap(cm_tot, annot=True, fmt="d", cmap="Blues")
plt.show()


In [ ]:
# ===== METHOD 1 — MEAN FIRING RATE PER SOGGETTO =====

X_raw = []

for subj in subjects:
    spikes_list = subject_to_spike_windows[subj]
    fr_all = [sp.mean(axis=0) for sp in spikes_list]
    X_raw.append(np.mean(fr_all, axis=0))

X_raw = np.array(X_raw)        # shape = (88 × 1008)
#y = labels

# === PCA COMPLETA ===
pca = PCA()
#pca.fit(X_raw)

X_pca = pca.fit_transform(X_raw)

var = np.cumsum(pca.explained_variance_ratio_)
k = np.searchsorted(var, SOGLIA_VAR) + 1
print("Componenti usati:", k)
X_pca = X_pca[:, :k]
# === SVM CROSS-FOLD ===
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
conf_mats = []
model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        gamma="scale",
        C=1.0
    ))
])
model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.85, svd_solver="full")),
    ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", gamma="scale", C=1.0))
])

conf_mats=[]
#for tr,ts in skf.split(X_pca,y):
 #   model=SVC(kernel="rbf",probability=True,class_weight="balanced")
 #   model.fit(X_pca[tr],y[tr])
 #   y_pred=model.predict(X_pca[ts])
 #   conf_mats.append(confusion_matrix(y[ts],y_pred))
for tr, ts in skf.split(X_raw, y):
    model.fit(X_raw[tr], y[tr])
    y_pred = model.predict(X_raw[ts])
    conf_mats.append(confusion_matrix(y[ts],y_pred))

cm_tot, rep_tot = compute_global_report(conf_mats)

print("=== MEAN FIRING RATE ===")
print(rep_tot)

plt.figure(figsize=(5,4))
sns.heatmap(cm_tot, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix Aggregata – Mean FR")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# ===== METHOD 2 — MEAN + VAR FIRING RATE =====

X_raw = []

for subj in subjects:
    spikes_list = subject_to_spike_windows[subj]
    feats = []
    for sp in spikes_list:
        fr = sp.mean(axis=0)
        vr = sp.var(axis=0)
        feats.append(np.concatenate([fr, vr]))
    X_raw.append(np.mean(feats, axis=0))

X_raw = np.array(X_raw)    # shape = (88 × 2016)
#y = labels

# PCA COMPLETA
pca = PCA()
#pca.fit(X_raw)

X_pca = pca.fit_transform(X_raw)

var = np.cumsum(pca.explained_variance_ratio_)
k = np.searchsorted(var, SOGLIA_VAR) + 1
X_pca = X_pca[:, :k]
print(f'Number of component: {k}')

model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        gamma="scale",
        C=1.0
    ))
])
# SVM CROSS FOLD
model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.85, svd_solver="full")),
    ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", gamma="scale", C=1.0))
])

conf_mats=[]
#for tr,ts in skf.split(X_pca,y):
 #   model=SVC(kernel="rbf",probability=True,class_weight="balanced")
 #   model.fit(X_pca[tr],y[tr])
 #   y_pred=model.predict(X_pca[ts])
 #   conf_mats.append(confusion_matrix(y[ts],y_pred))
for tr, ts in skf.split(X_raw, y):
    model.fit(X_raw[tr], y[tr])
    y_pred = model.predict(X_raw[ts])
    conf_mats.append(confusion_matrix(y[ts],y_pred))

cm_tot, rep_tot = compute_global_report(conf_mats)
print("=== MEAN + VAR FIRING RATE ===")
print(rep_tot)

plt.figure(figsize=(5,4))
sns.heatmap(cm_tot, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix Aggregata – Mean+Var FR")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.show()


In [ ]:
# ===== METHOD 3 — TEMPORAL BINNING =====

BINS = 20
bin_edges = np.linspace(0,1,BINS+1)

def extract_binning(spikes):
    T, N = spikes.shape
    time_norm = np.linspace(0,1,T)
    feat=[]
    for i in range(N):
        spike_t = time_norm[spikes[:,i] > 0]
        h,_ = np.histogram(spike_t, bins=bin_edges)
        feat.extend(h)
    return np.array(feat)

X_raw=[]
for subj in subjects:
    spikes_list = subject_to_spike_windows[subj]
    feats = [extract_binning(sp) for sp in spikes_list]
    X_raw.append(np.mean(feats, axis=0))

X_raw=np.array(X_raw)
#y=labels

pca=PCA()
#pca.fit(X_raw)


X_pca=pca.fit_transform(X_raw)
var=np.cumsum(pca.explained_variance_ratio_)
k=np.searchsorted(var,SOGLIA_VAR)+1
print(f'Number of component: {k}')
X_pca = X_pca[:, :k]
model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        gamma="scale",
        C=1.0
    ))
])
conf_mats=[]
subject_pred = {s: [] for s in subjects}   # lista predizioni per soggetto
subject_true = {s: subject_labels[s] for s in subjects}

model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.85, svd_solver="full")),
    ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", gamma="scale", C=1.0))
])

conf_mats=[]
#for tr,ts in skf.split(X_pca,y):
 #   model=SVC(kernel="rbf",probability=True,class_weight="balanced")
 #   model.fit(X_pca[tr],y[tr])
 #   y_pred=model.predict(X_pca[ts])
 #   conf_mats.append(confusion_matrix(y[ts],y_pred))
for tr, ts in skf.split(X_raw, y):
    model.fit(X_raw[tr], y[tr])
    y_pred = model.predict(X_raw[ts])
    conf_mats.append(confusion_matrix(y[ts],y_pred))
    # aggiungi predizione corretta al soggetto corrispondente
    for idx, pred in zip(ts, y_pred):
        subj = subjects[idx]
        subject_pred[subj].append(pred)

cm_tot,rep_tot=compute_global_report(conf_mats)
print("=== TEMPORAL BINNING ===")
print(rep_tot)

plt.figure(figsize=(5,4))
sns.heatmap(cm_tot,annot=True,fmt="d",cmap="Blues")
plt.title("Confusion Matrix – Binning")
plt.xlabel("Predicted");plt.ylabel("True")
plt.show()

final_pred = {s: int(np.round(np.mean(subject_pred[s]))) for s in subjects}

# soggetti sbagliati
wrong_subjects = [s for s in subjects if final_pred[s] != subject_true[s]]

print("\n=== SOGGETTI SBAGLIATI ===")
for s in wrong_subjects:
    print(f"{s}: predicted={final_pred[s]}, true={subject_true[s]}")
print("\n=== SOGGETTI SBAGLIATI (Metodo 3 – Temporal Binning) ===")

In [ ]:
# ===== METHOD 4 — FLATTEN T×N =====

X_raw=[]
for subj in subjects:
    windows=subject_to_spike_windows[subj]
    feats=[sp.flatten() for sp in windows]
    X_raw.append(np.mean(feats,axis=0))

X_raw=np.array(X_raw)
#y=labels

pca=PCA()
#pca.fit(X_raw)
X_pca=pca.fit_transform(X_raw)
var=np.cumsum(pca.explained_variance_ratio_)
k=np.searchsorted(var,SOGLIA_VAR)+1
print(f'Number of component: {k}')

X_pca = X_pca[:, :k]
model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        gamma="scale",
        C=1.0
    ))
])
model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.85, svd_solver="full")),
    ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", gamma="scale", C=1.0))
])

conf_mats=[]
#for tr,ts in skf.split(X_pca,y):
 #   model=SVC(kernel="rbf",probability=True,class_weight="balanced")
 #   model.fit(X_pca[tr],y[tr])
 #   y_pred=model.predict(X_pca[ts])
 #   conf_mats.append(confusion_matrix(y[ts],y_pred))
for tr, ts in skf.split(X_raw, y):
    model.fit(X_raw[tr], y[tr])
    y_pred = model.predict(X_raw[ts])
    conf_mats.append(confusion_matrix(y[ts],y_pred))

cm_tot,rep_tot=compute_global_report(conf_mats)
print("=== FLATTEN T×N ===")
print(rep_tot)

plt.figure(figsize=(5,4))
sns.heatmap(cm_tot,annot=True,fmt="d",cmap="Blues")
plt.title("Confusion Matrix – Flatten T×N")
plt.xlabel("Predicted");plt.ylabel("True")
plt.show()


In [ ]:
# ===== METHOD 5 — POPULATION ACTIVITY =====

def pop_features(sp):
    pop = sp.sum(axis=1)
    feats = [
        pop.mean(),
        pop.std(),
        pop.max(),
        pop.min(),
        pop.var(),
        np.median(pop),
        np.percentile(pop,90),
        np.percentile(pop,10),
    ]
    return np.array(feats)

X_raw=[]
for subj in subjects:
    feats=[pop_features(sp) for sp in subject_to_spike_windows[subj]]
    X_raw.append(np.mean(feats,axis=0))

X_raw=np.array(X_raw)
#y=labels

pca=PCA()
X_pca=pca.fit_transform(X_raw)
var=np.cumsum(pca.explained_variance_ratio_)
k=np.searchsorted(var,SOGLIA_VAR)+1
X_pca = X_pca[:, :k]
print(f'Number of component: {k}')

model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        gamma="scale",
        C=1.0
    ))
])
model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.85, svd_solver="full")),
    ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", gamma="scale", C=1.0))
])

conf_mats=[]
#for tr,ts in skf.split(X_pca,y):
 #   model=SVC(kernel="rbf",probability=True,class_weight="balanced")
 #   model.fit(X_pca[tr],y[tr])
 #   y_pred=model.predict(X_pca[ts])
 #   conf_mats.append(confusion_matrix(y[ts],y_pred))
for tr, ts in skf.split(X_raw, y):
    model.fit(X_raw[tr], y[tr])
    y_pred = model.predict(X_raw[ts])
    conf_mats.append(confusion_matrix(y[ts],y_pred))

cm_tot,rep_tot=compute_global_report(conf_mats)
print("=== POPULATION ACTIVITY ===")
print(rep_tot)

plt.figure(figsize=(5,4))
sns.heatmap(cm_tot,annot=True,fmt="d",cmap="Blues")
plt.title("Confusion Matrix – Population")
plt.xlabel("Predicted");plt.ylabel("True")
plt.show()


In [ ]:
# ===== METHOD 6 — CROSS-CORR LIGHT =====

def ccorr_light(sp, max_pairs=30):
    T,N=sp.shape
    idx=np.random.choice(N, max_pairs, replace=False)
    feats=[]
    for i in range(max_pairs-1):
        c=np.correlate(sp[:,idx[i]], sp[:,idx[i+1]])
        feats.append(c[0])
    return np.array(feats)

X_raw=[]
for subj in subjects:
    feats=[ccorr_light(sp) for sp in subject_to_spike_windows[subj]]
    X_raw.append(np.mean(feats,axis=0))

X_raw=np.array(X_raw)
#y=labels
\
pca=PCA()
X_pca=pca.fit_transform(X_raw)
var=np.cumsum(pca.explained_variance_ratio_)
k=np.searchsorted(var,SOGLIA_VAR)+1
X_pca = X_pca[:, :k]
print(f'Number of component: {k}')

model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        gamma="scale",
        C=1.0
    ))
])
model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.85, svd_solver="full")),
    ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", gamma="scale", C=1.0))
])

conf_mats=[]
#for tr,ts in skf.split(X_pca,y):
 #   model=SVC(kernel="rbf",probability=True,class_weight="balanced")
 #   model.fit(X_pca[tr],y[tr])
 #   y_pred=model.predict(X_pca[ts])
 #   conf_mats.append(confusion_matrix(y[ts],y_pred))
for tr, ts in skf.split(X_raw, y):
    model.fit(X_raw[tr], y[tr])
    y_pred = model.predict(X_raw[ts])
    conf_mats.append(confusion_matrix(y[ts],y_pred))

cm_tot,rep_tot=compute_global_report(conf_mats)
print("=== CROSS-CORR LIGHT ===")
print(rep_tot)

plt.figure(figsize=(5,4))
sns.heatmap(cm_tot,annot=True,fmt="d",cmap="Blues")
plt.title("Confusion Matrix – CCorr")
plt.xlabel("Predicted");plt.ylabel("True")
plt.show()


In [ ]:
# ===== METHOD 7 — HISTOGRAM 2D POOLING =====

BINS=9 #9 xkè neuroni 3x3
bin_edges=np.linspace(0,1,BINS+1)

def hist_pool(sp):
    T,N=sp.shape
    time_norm=np.linspace(0,1,T)
    feat=[]
    for i in range(N):
        spike_t=time_norm[sp[:,i]>0]
        h,_=np.histogram(spike_t, bins=bin_edges)
        feat.extend(h)
    return np.array(feat)
print(f'Number of component: {k}')

X_raw=[]
for subj in subjects:
    feats=[hist_pool(sp) for sp in subject_to_spike_windows[subj]]
    X_raw.append(np.mean(feats,axis=0))

X_raw=np.array(X_raw)
#y=labels

pca=PCA()
X_pca=pca.fit_transform(X_raw)
var=np.cumsum(pca.explained_variance_ratio_)
k=np.searchsorted(var,SOGLIA_VAR)+1
X_pca = X_pca[:, :k]

model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        gamma="scale",
        C=1.0
    ))
])

model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.85, svd_solver="full")),
    ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", gamma="scale", C=1.0))
])

conf_mats=[]
#for tr,ts in skf.split(X_pca,y):
 #   model=SVC(kernel="rbf",probability=True,class_weight="balanced")
 #   model.fit(X_pca[tr],y[tr])
 #   y_pred=model.predict(X_pca[ts])
 #   conf_mats.append(confusion_matrix(y[ts],y_pred))
for tr, ts in skf.split(X_raw, y):
    model.fit(X_raw[tr], y[tr])
    y_pred = model.predict(X_raw[ts])
    conf_mats.append(confusion_matrix(y[ts],y_pred))
    
cm_tot,rep_tot=compute_global_report(conf_mats)
print("=== HISTOGRAM POOLING ===")
print(rep_tot)

plt.figure(figsize=(5,4))
sns.heatmap(cm_tot,annot=True,fmt="d",cmap="Blues")
plt.title("Confusion Matrix – Hist2D")
plt.xlabel("Predicted");plt.ylabel("True")
plt.show()
